# 🔬 GlassBox: Zero-Overhead Mechanistic Interpretability & Live Benchmark

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/azzouzabdelhak68-ship-it/glassbox/blob/main/notebooks/quickstart_demo.ipynb)
[![GitHub](https://img.shields.io/badge/GitHub-GlassBox-blue.svg?logo=github)](https://github.com/azzouzabdelhak68-ship-it/glassbox)
[![PyTorch 2.4+](https://img.shields.io/badge/PyTorch-2.4%2B-EE4C2C?logo=pytorch&logoColor=white)](https://pytorch.org)
[![License: Apache 2.0](https://img.shields.io/badge/License-Apache%202.0-green.svg)](LICENSE)

**GlassBox** is a zero-overhead flight data recorder and mechanistic interpretability engine for PyTorch LLMs. Traditional observability and interpretability hooks introduce crippling GPU synchronization stalls (`.cpu()` / `.item()`) that degrade throughput by 15–50%. GlassBox uses non-blocking asynchronous CUDA DMA copies to pinned host memory, achieving **<1% overhead** while recording tamper-evident mathematical evidence (SHA-256 seal) of internal neuron firings, attention dynamics, and residual streams.

### What this interactive notebook covers:
1. ⚡ **1-Click Automated Setup**: Installs GlassBox and downloads an ultra-small open-source LLM (`Qwen/Qwen2.5-0.5B` with `gpt2` fallback).
2. 🎛️ **Interactive Tracing Configuration**: Custom selection of prompts, layers, and neuron coordinates to trace.
3. ⏱️ **Live 3-Way Benchmark**: Plain PyTorch baseline vs. Naive Synchronous Hook vs. GlassBox Async DMA Flight Recorder.
4. 🔭 **Mechanistic Interpretability**:
   - **Logit Lens**: Layer-by-layer predicted tokens and confidence progression.
   - **Top-Activating Neurons**: Identifies and inspects the most active neuron coordinates per token.
   - **Embedded Calm Viewer**: Standalone, interactive HTML report with inline SVG heatmaps rendered directly in Colab.

In [ ]:
# @title 📦 1. Automated Environment Setup & Dependency Installation
import os
import sys

print("📦 Installing GlassBox and core dependencies...")
# In Google Colab, install glassbox and dependencies via git
!pip install -q \
    git+https://github.com/azzouzabdelhak68-ship-it/glassbox.git \
    transformers torch accelerate matplotlib pandas IPython

# Add local path support when running within a cloned repository
if os.path.exists("../glassbox") and ".." not in sys.path:
    sys.path.insert(0, "..")
elif os.path.exists("./glassbox") and "." not in sys.path:
    sys.path.insert(0, ".")

import torch  # noqa: E402
import transformers  # noqa: E402

import glassbox  # noqa: E402

print(f"✅ PyTorch version: {torch.__version__}")
print(f"✅ Transformers version: {transformers.__version__}")
print(f"✅ GlassBox version: {glassbox.__version__}")

device = "cuda" if torch.cuda.is_available() else "cpu"
device_name = torch.cuda.get_device_name(0) if device == "cuda" else "Standard Colab CPU"
print(f"🚀 Execution Device: {device.upper()} ({device_name})")

## 2. Ultra-Small Open-Source LLM Initialization

We download and initialize an ultra-small, fast open-weights model:
- **Primary:** `Qwen/Qwen2.5-0.5B` (~500M parameters, modern SwiGLU + RoPE architecture, runs in seconds on free Colab CPU or T4 GPU).
- **Fallback:** `gpt2` (124M parameters, classic architecture).

We also extract the model's exact architecture inventory to dynamically configure GlassBox.

In [ ]:
# @title 🤖 2. Download & Initialize Lightweight Open-Weights LLM
from transformers import AutoModelForCausalLM, AutoTokenizer


def load_demo_llm():
    """Downloads and initializes an ultra-small, fast open-weights LLM.

    Tries Qwen/Qwen2.5-0.5B first; falls back to EleutherAI/pythia-70m or gpt2 if needed.
    """
    candidates = [
        "Qwen/Qwen2.5-0.5B",
        "EleutherAI/pythia-70m",
        "gpt2",
    ]

    device = "cuda" if torch.cuda.is_available() else "cpu"
    dtype = torch.float16 if device == "cuda" else torch.float32

    for model_id in candidates:
        try:
            print(f"📥 Loading model '{model_id}'...")
            tokenizer = AutoTokenizer.from_pretrained(model_id)
            if tokenizer.pad_token is None:
                tokenizer.pad_token = tokenizer.eos_token

            model = AutoModelForCausalLM.from_pretrained(
                model_id,
                torch_dtype=dtype,
                low_cpu_mem_usage=True,
            ).to(device)
            model.eval()
            print(f"✨ Successfully loaded '{model_id}' on {device.upper()} ({dtype})!")
            return model, tokenizer, model_id, device, dtype
        except Exception as err:
            print(f"⚠️ Notice: could not load '{model_id}': {err}. Trying fallback...")

    raise RuntimeError("Could not load any candidate open-weights model.")


model, tokenizer, MODEL_NAME, DEVICE, DTYPE = load_demo_llm()

# Extract architecture parameters robustly across diverse model configs
config = model.config
NUM_LAYERS = getattr(config, "num_hidden_layers", None) or getattr(config, "n_layer", None) or 12
HIDDEN_SIZE = getattr(config, "hidden_size", None) or getattr(config, "n_embd", None) or 768
NUM_HEADS = getattr(config, "num_attention_heads", None) or getattr(config, "n_head", None) or 12
INTERMEDIATE_SIZE = (
    getattr(config, "intermediate_size", None)
    or getattr(config, "n_inner", None)
    or (4 * HIDDEN_SIZE)
)
VOCAB_SIZE = getattr(config, "vocab_size", 50257) or 50257

print("\n📊 Model Architecture Inventory:")
print(f"   • Layers: {NUM_LAYERS}")
print(f"   • Hidden Dimension: {HIDDEN_SIZE}")
print(f"   • Attention Heads: {NUM_HEADS}")
print(f"   • MLP Intermediate Neurons: {INTERMEDIATE_SIZE}")
print(f"   • Vocabulary Size: {VOCAB_SIZE}")

## 3. Interactive Configuration: Prompts, Layers & Neurons

GlassBox uses a declarative coordinate addressing DSL:
- `N-LL-NNNN`: Specific MLP intermediate post-activation neuron `1..intermediate_size` in layer `LL`.
- `A-LL-HH`: Attention head `HH` in layer `LL` (computed via on-device L2 norm reduction across head dimension).
- `L-LL`: Residual stream representation at layer `LL` (L2 norm across hidden dimension).
- `LOGITS`: Top-1 logit at the vocabulary unembedding head.

Use the form below to customize the prompt, generation length, and which internal components to trace.

In [ ]:
# @title ⚙️ 3. Interactive Configuration: Select Prompts & Watchlist Coordinates
import yaml

# Interactive prompt and generation controls
TEST_PROMPT = "The capital of France is"  # @param {type:"string"}
GENERATION_TOKENS = 20  # @param {type:"slider", min:8, max:64, step:4}

# Select layers to trace (1-indexed)
TRACE_LAYERS = [1, 2, 4, 8] if NUM_LAYERS >= 8 else [1, 2]
NEURONS_PER_LAYER = 32  # @param {type:"slider", min:8, max:128, step:8}
INCLUDE_ATTENTION_HEADS = True  # @param {type:"boolean"}
INCLUDE_RESIDUAL_STREAMS = True  # @param {type:"boolean"}
INCLUDE_LOGITS = True  # @param {type:"boolean"}

# Build addresses dynamically
watched_addresses = []
for layer_idx in TRACE_LAYERS:
    if layer_idx <= NUM_LAYERS:
        # MLP intermediate neuron coordinate range
        max_n = min(NEURONS_PER_LAYER, INTERMEDIATE_SIZE)
        watched_addresses.append(f"N-{layer_idx:02d}-0001..{max_n:04d}")

        # Residual stream layer norm
        if INCLUDE_RESIDUAL_STREAMS:
            watched_addresses.append(f"L-{layer_idx:02d}")

        # Attention head L2 norms
        if INCLUDE_ATTENTION_HEADS:
            max_h = min(4, NUM_HEADS)
            watched_addresses.append(f"A-{layer_idx:02d}-01..{max_h:02d}")

if INCLUDE_LOGITS:
    watched_addresses.append("LOGITS")

# Synthesize GlassBox Model Manifest
manifest = {
    "model_id": MODEL_NAME.replace("/", "_"),
    "checkpoint_sha256": "0" * 64,
    "architecture": {
        "layers": NUM_LAYERS,
        "hidden": HIDDEN_SIZE,
        "heads": NUM_HEADS,
        "mlp_per_layer": INTERMEDIATE_SIZE,
        "vocab": VOCAB_SIZE,
    },
}

# Generate Declarative Watchlist YAML
watchlist_path = "quickstart_watchlist.yaml"
watchlist_dict = {
    "version": 1,
    "model_id": manifest["model_id"],
    "checkpoint_sha256": manifest["checkpoint_sha256"],
    "watchlist_id": "colab_interactive_demo",
    "trigger": "always",
    "window": {
        "start_token": 0,
        "end_token": 512,
        "every_n": 1,
    },
    "precision": "fp16",
    "entries": [{"addresses": watched_addresses}],
}

with open(watchlist_path, "w", encoding="utf-8") as f:
    yaml.safe_dump(watchlist_dict, f, sort_keys=False)

print(f"📋 Watchlist successfully compiled to '{watchlist_path}':")
print(f"   • Target Prompt: '{TEST_PROMPT}'")
print(f"   • Generating: {GENERATION_TOKENS} new tokens")
print(f"   • Traced Address Patterns ({len(watched_addresses)}): {watched_addresses}")

## 4. Live Performance Benchmark: 3-Way Comparison

We benchmark inference throughput and latency across three conditions:
1. **Plain PyTorch Baseline**: Un-instrumented model inference.
2. **Naive PyTorch Hook (`.cpu()`)**: Standard interpretability approach copying activations to CPU synchronously inside forward hooks. On GPU, this introduces a **GPU synchronization bubble** on every single token step, draining the execution pipeline.
3. **GlassBox Async DMA**: Instruments the model with dedicated pinned memory double buffers and zero-synchronization asynchronous DMA transfers, achieving **<1% overhead**.

In [ ]:
# @title ⏱️ 4. Run Live 3-Way Performance Benchmark
import time

import pandas as pd

from glassbox import GlassBox

input_ids = tokenizer(TEST_PROMPT, return_tensors="pt")["input_ids"].to(DEVICE)
prompt_len = input_ids.shape[1]
BENCHMARK_ITERS = 5


def run_inference(m, in_ids, max_new):
    """Generates max_new tokens autoregressively."""
    with torch.no_grad():
        out = m.generate(
            in_ids,
            max_new_tokens=max_new,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )
    return out


# Warmup run to initialize CUDA context / PyTorch caching
print("🔥 Warming up model pipeline...")
_ = run_inference(model, input_ids, 4)
if DEVICE == "cuda":
    torch.cuda.synchronize()

results = []

# --- 1. Baseline: Plain PyTorch (No Instrumentation) ---
print("\n[1/3] Benchmarking Baseline PyTorch (un-instrumented)...")
t_start = time.perf_counter()
for _ in range(BENCHMARK_ITERS):
    _ = run_inference(model, input_ids, GENERATION_TOKENS)
    if DEVICE == "cuda":
        torch.cuda.synchronize()
t_baseline_total = (time.perf_counter() - t_start) / BENCHMARK_ITERS
baseline_tokens_sec = GENERATION_TOKENS / t_baseline_total
baseline_ms_per_tok = (t_baseline_total / GENERATION_TOKENS) * 1000.0

results.append(
    {
        "Condition": "1. Baseline PyTorch",
        "Total Time (s)": round(t_baseline_total, 4),
        "Throughput (tok/s)": round(baseline_tokens_sec, 2),
        "Latency/Token (ms)": round(baseline_ms_per_tok, 2),
        "Overhead (%)": 0.0,
    }
)

# --- 2. Naive PyTorch Hook (.cpu() sync stall) ---
print("[2/3] Benchmarking Naive Hook (.cpu() sync stall)...")
naive_handles = []


def _make_naive():
    def _hook(module, inp, out):
        val = out[0] if isinstance(out, tuple) else out
        # Synchronous host barrier stall
        _ = val.detach().cpu().numpy()

    return _hook


for name, mod in model.named_modules():
    if any(
        name.endswith(f"layers.{layer_num - 1}.mlp") or name.endswith(f"h.{layer_num - 1}.mlp")
        for layer_num in TRACE_LAYERS
    ):
        naive_handles.append(mod.register_forward_hook(_make_naive()))

if not naive_handles:
    for name, mod in model.named_modules():
        if (
            any(f"layers.{l_num - 1}" in name or f"h.{l_num - 1}" in name for l_num in TRACE_LAYERS)
            and "mlp" in name
        ):
            naive_handles.append(mod.register_forward_hook(_make_naive()))
            break

try:
    t_start = time.perf_counter()
    for _ in range(BENCHMARK_ITERS):
        _ = run_inference(model, input_ids, GENERATION_TOKENS)
        if DEVICE == "cuda":
            torch.cuda.synchronize()
    t_naive_total = (time.perf_counter() - t_start) / BENCHMARK_ITERS
finally:
    for h in naive_handles:
        h.remove()

naive_tokens_sec = GENERATION_TOKENS / t_naive_total
naive_ms_per_tok = (t_naive_total / GENERATION_TOKENS) * 1000.0
naive_overhead = ((t_naive_total - t_baseline_total) / t_baseline_total) * 100.0

results.append(
    {
        "Condition": "2. Naive PyTorch Hook (.cpu())",
        "Total Time (s)": round(t_naive_total, 4),
        "Throughput (tok/s)": round(naive_tokens_sec, 2),
        "Latency/Token (ms)": round(naive_ms_per_tok, 2),
        "Overhead (%)": round(naive_overhead, 2),
    }
)

# --- 3. GlassBox Async DMA Flight Recorder ---
print("[3/3] Benchmarking GlassBox Flight Recorder (<1% async DMA overhead)...")
t_start = time.perf_counter()
last_run_id = None
generated_tokens_output = None

for _ in range(BENCHMARK_ITERS):
    with GlassBox.watch(
        watchlist_path,
        model=model,
        manifest=manifest,
        prompt=TEST_PROMPT,
    ) as ctx:
        out_ids = run_inference(model, input_ids, GENERATION_TOKENS)
        if DEVICE == "cuda":
            torch.cuda.synchronize()
        generated_tokens_output = out_ids
    # Sealed run_id is finalized upon exiting the GlassBox.watch context
    last_run_id = ctx.run_id

t_glassbox_total = (time.perf_counter() - t_start) / BENCHMARK_ITERS
glassbox_tokens_sec = GENERATION_TOKENS / t_glassbox_total
glassbox_ms_per_tok = (t_glassbox_total / GENERATION_TOKENS) * 1000.0
glassbox_overhead = ((t_glassbox_total - t_baseline_total) / t_baseline_total) * 100.0

results.append(
    {
        "Condition": "3. GlassBox Async DMA",
        "Total Time (s)": round(t_glassbox_total, 4),
        "Throughput (tok/s)": round(glassbox_tokens_sec, 2),
        "Latency/Token (ms)": round(glassbox_ms_per_tok, 2),
        "Overhead (%)": round(glassbox_overhead, 2),
    }
)

# Render Summary Table
benchmark_df = pd.DataFrame(results)
print("\n" + "=" * 70)
print("🏆 LIVE BENCHMARK SUMMARY TABLE")
print("=" * 70)
display(benchmark_df)

full_output_text = tokenizer.decode(generated_tokens_output[0], skip_special_tokens=True)
print(f'\n💬 Model Completion: "{full_output_text}"')

In [ ]:
# @title 📊 5. Plot Benchmark Comparison Chart
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4.8), dpi=100)

labels = ["Baseline\n(Plain PyTorch)", "Naive Hook\n(.cpu() sync stall)", "GlassBox\n(Async DMA)"]
throughputs = [r["Throughput (tok/s)"] for r in results]
latencies = [r["Latency/Token (ms)"] for r in results]
colors = ["#3b82f6", "#ef4444", "#10b981"]

# 1. Throughput Chart
bars1 = ax1.bar(labels, throughputs, color=colors, width=0.55, edgecolor="#1f2937", linewidth=1.2)
ax1.set_title("Inference Throughput (Tokens / Sec)", fontsize=12, fontweight="bold", pad=12)
ax1.set_ylabel("Tokens / Sec (Higher is Better)", fontsize=10)
ax1.grid(axis="y", linestyle="--", alpha=0.5)

y_offset_tp = max(throughputs) * 0.02
for bar in bars1:
    y = bar.get_height()
    ax1.text(
        bar.get_x() + bar.get_width() / 2.0,
        y + y_offset_tp,
        f"{y:.1f} t/s",
        ha="center",
        va="bottom",
        fontweight="bold",
        fontsize=10,
    )

# 2. Latency Chart
bars2 = ax2.bar(labels, latencies, color=colors, width=0.55, edgecolor="#1f2937", linewidth=1.2)
ax2.set_title("Latency per Token (ms)", fontsize=12, fontweight="bold", pad=12)
ax2.set_ylabel("Milliseconds / Token (Lower is Better)", fontsize=10)
ax2.grid(axis="y", linestyle="--", alpha=0.5)

y_offset_lat = max(latencies) * 0.02
for bar in bars2:
    y = bar.get_height()
    ax2.text(
        bar.get_x() + bar.get_width() / 2.0,
        y + y_offset_lat,
        f"{y:.1f} ms",
        ha="center",
        va="bottom",
        fontweight="bold",
        fontsize=10,
    )

plt.suptitle(
    f"GlassBox Zero-Overhead Telemetry vs Naive Instrumentation on {MODEL_NAME}",
    fontsize=13,
    fontweight="bold",
    y=1.02,
)
plt.tight_layout()
plt.show()

## 5. Mechanistic Interpretability & Neuron Activation Analysis

Now let's inspect the telemetry captured by GlassBox:
1. **Cryptographic Verification**: Proves the mathematical activations in the flight store have not been tampered with.
2. **Logit Lens**: Projects intermediate residual representations directly through the unembedding head to reveal layer-by-layer predicted tokens and confidence.
3. **Top-Activating Neurons**: Identifies which specific MLP intermediate neuron coordinates fired with the highest magnitude for each token in the prompt.

In [ ]:
# @title 🔭 6. Mechanistic Logit Lens & Residual Stream Progression
import pandas as pd
import torch

from glassbox import GlassBox
from glassbox.lens import format_lens_table

print(f"🔒 Sealed Run ID: {last_run_id}")
is_verified = GlassBox.verify(last_run_id)
print(f"🛡️ Cryptographic SHA-256 Provenance Chain Verified: {is_verified}")

# 1. GlassBox Flight Recorder Native Lens
lens_result = GlassBox.lens(last_run_id, token_pos=0)
print("\n" + "=" * 65)
print("🔭 GlassBox Flight Telemetry Residual Stream Progression")
print("=" * 65)
print(format_lens_table(lens_result))

# 2. Detailed Vocabulary Prediction across Layers
print("\n" + "=" * 65)
print("🧠 Logit Lens: Layer-by-Layer Predicted Tokens & Probabilities")
print("=" * 65)


def analyze_logit_lens_predictions(m, tok, text, layers_to_inspect=None):
    """Projects intermediate residual stream vectors through the final norm and unembedding head
    to demonstrate layer-by-layer token predictions as the prompt progresses.
    """
    inputs = tok(text, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        outputs = m(**inputs, output_hidden_states=True)

    hidden_states = outputs.hidden_states  # layer 0 = embeddings, 1..N = layers
    tokens = inputs["input_ids"][0]
    num_prompt_tokens = len(tokens)

    # Locate norm and lm_head across model architectures
    lm_head = (
        getattr(m, "lm_head", None) or getattr(m, "embed_out", None) or getattr(m, "output", None)
    )
    norm = (
        getattr(getattr(m, "model", m), "norm", None)
        or getattr(getattr(m, "transformer", None), "ln_f", None)
        or getattr(getattr(m, "gpt_neox", None), "final_layer_norm", None)
    )

    layer_indices = layers_to_inspect or list(range(1, len(hidden_states)))

    # A) Matrix view: progression across all prompt tokens
    print(f"Prompt: '{text}' ({num_prompt_tokens} tokens)")
    print("Layer-by-Layer Next-Token Predictions As Prompt Progresses:\n")

    grid_data = {}
    for pos in range(num_prompt_tokens):
        pos_token_str = tok.decode([tokens[pos]]).strip() or f"tok_{pos}"
        col_label = f"#{pos} [{pos_token_str}]"
        col_preds = []
        for l_idx in layer_indices:
            if l_idx < len(hidden_states):
                h_state = hidden_states[l_idx][:, pos : pos + 1, :]
                h_normed = norm(h_state) if norm is not None else h_state
                if lm_head is not None:
                    logits = lm_head(h_normed)[0, 0]
                    probs = torch.softmax(logits.float(), dim=-1)
                    top_prob, top_id = torch.topk(probs, k=1)
                    pred_tok = tok.decode([top_id.item()]).replace("\n", "\\n")
                    col_preds.append(f"{pred_tok} ({top_prob.item() * 100:.1f}%)")
                else:
                    col_preds.append("N/A")
        grid_data[col_label] = col_preds

    row_labels = [f"L-{l_idx:02d}" for l_idx in layer_indices if l_idx < len(hidden_states)]
    progression_df = pd.DataFrame(grid_data, index=row_labels)
    display(progression_df)

    # B) Detailed table for the final completion prediction
    target_pos = num_prompt_tokens - 1
    last_prompt_tok = tok.decode([tokens[target_pos]])
    print(
        f"\n🎯 Target Token #{target_pos} ({repr(last_prompt_tok)}) -> "
        "Layer-by-layer predicted completion:"
    )
    records = []
    for l_idx in layer_indices:
        if l_idx < len(hidden_states):
            h_state = hidden_states[l_idx][:, target_pos : target_pos + 1, :]
            h_normed = norm(h_state) if norm is not None else h_state
            if lm_head is not None:
                logits = lm_head(h_normed)[0, 0]
                probs = torch.softmax(logits.float(), dim=-1)
                top_prob, top_id = torch.topk(probs, k=1)
                pred_token = tok.decode([top_id.item()]).replace("\n", "\\n")
                records.append(
                    {
                        "Layer": f"L-{l_idx:02d}",
                        "Top Predicted Token": repr(pred_token),
                        "Confidence (%)": round(top_prob.item() * 100.0, 2),
                        "Top Token ID": top_id.item(),
                    }
                )
    display(pd.DataFrame(records))


analyze_logit_lens_predictions(model, tokenizer, TEST_PROMPT, TRACE_LAYERS)

In [ ]:
# @title ⚡ 7. Top-Activating Neuron / Feature Inspection
df = GlassBox.to_dataframe(last_run_id)

print(f"📊 Total Captured Telemetry Records: {df.shape[0]} rows")

# Filter for MLP intermediate neurons (N-LL-NNNN)
neuron_df = df[df["address"].str.startswith("N-")].copy()

if not neuron_df.empty:
    print("\n" + "=" * 70)
    print("⚡ TOP-ACTIVATING NEURONS PER TOKEN IN PROMPT")
    print("=" * 70)

    prompt_tokens = input_ids[0].tolist()

    for pos, group in neuron_df.groupby("token_pos"):
        if pos < len(prompt_tokens):
            tok_str = tokenizer.decode([prompt_tokens[pos]])
            header_prefix = f"Prompt Token #{pos}"
        else:
            gen_idx = pos - len(prompt_tokens) + 1
            tok_str = f"Generated #{gen_idx}"
            header_prefix = f"Generated Token #{gen_idx}"

        print(f"\n📍 {header_prefix} -> {repr(tok_str)}:")

        # Sort by absolute activation value to find most decisive neurons
        top_neurons = group.sort_values(by="value", ascending=False, key=abs).head(4)
        for _, row in top_neurons.iterrows():
            addr = row["address"]
            val = row["value"]
            parts = addr.split("-")
            layer_num, neuron_id = int(parts[1]), int(parts[2])
            sign = "📈 (+)" if val > 0 else "📉 (-)"
            print(f"   • {addr} [Layer {layer_num:02d}, Neuron {neuron_id:04d}]: {val:+.4f} {sign}")

## 6. Interactive Calm Viewer Integration

GlassBox includes the **Calm Viewer**: a 100% self-contained single-file HTML report featuring:
- **Interactive Token Strip**: Click any token to inspect its exact activations.
- **Inline SVG Heatmap**: Matrix of watched addresses vs tokens.
- **Sparklines & Z-scores**: Real-time variance indicators.
- **Bipartite Circuit Co-occurrence Graphs**: Visualizing synchronous neuron firings.
- **Cryptographic Seal Badge**: Visual proof of flight record integrity.

The cell below generates this HTML report, saves it for local download, and embeds it directly in the Colab output using an isolated iframe.

In [ ]:
# @title 🎨 8. Render Embedded Calm Viewer Report
import html

from IPython.display import HTML, display

from glassbox.viewer import render_html

# 1. Render self-contained static HTML report
calm_html = render_html(last_run_id)

# 2. Save file for offline download
report_path = f"glassbox_{last_run_id}_calm_viewer.html"
with open(report_path, "w", encoding="utf-8") as f:
    f.write(calm_html)

print(f"✅ Calm Viewer report successfully generated: {report_path} ({len(calm_html)} bytes)")
print("👉 You can download this file from Colab's left sidebar to view it offline in any browser.")
print("\n🖥️ Interactive Calm Viewer embedded directly below:")

# 3. Embed in notebook output using an isolated iframe
iframe_card = f"""
<div style="border: 2px solid #2563eb; border-radius: 8px; overflow: hidden; margin: 15px 0;">
  <div style="background: #2563eb; color: white; padding: 10px 16px; font-weight: bold;">
    <span>GlassBox Flight Store: Interactive Calm Viewer &amp; Circuit Inspector</span>
    <span style="float: right; font-size: 12px;">Run ID: {last_run_id}</span>
  </div>
  <iframe srcdoc="{html.escape(calm_html)}" style="width: 100%; height: 750px; border: none;">
  </iframe>
</div>
"""
display(HTML(iframe_card))

## 7. Token-Level Telemetry Export & Frontier-LLM Analysis Bundle

This section extracts fine-grained, token-by-token internal states and packages them for deep mechanistic analysis:
1. **Token-Level Telemetry (`token_level_telemetry.json`)**: Exact top-activating neuron coordinates and layer-by-layer Logit Lens predictions recorded for every individual token in your prompt.
2. **Frontier-LLM Ready Prompt (`frontier_llm_prompt.md`)**: A domain-agnostic, structured interpretability prompt ready to paste directly into **Claude 3.5 Sonnet, GPT-4o, or Gemini 1.5/2.0 Pro** to interpret neuron functions and circuit formation on any custom prompt.
3. **Complete Run Artifacts (`glassbox_artifacts_<RUN>.zip`)**: Includes the Calm Viewer HTML, benchmark metrics, telemetry data, and cryptographic seal.
4. **Export Options**: 1-click browser download, optional Google Drive auto-save, and 1-click sharing to the **GlassBox Community Discussions** tab.

In [ ]:
# @title 📦 9. Generate Token-Level Telemetry & Frontier LLM Analysis Prompt
import json
import os
import shutil
import zipfile

import pandas as pd
import torch

from glassbox import GlassBox

# 1. Forward pass to extract hidden states for token-level Logit Lens
inputs = tokenizer(TEST_PROMPT, return_tensors="pt").to(DEVICE)
prompt_token_ids = inputs["input_ids"][0].tolist()
num_tokens = len(prompt_token_ids)

with torch.no_grad():
    outputs = model(**inputs, output_hidden_states=True)
hidden_states = outputs.hidden_states

lm_head = (
    getattr(model, "lm_head", None)
    or getattr(model, "embed_out", None)
    or getattr(model, "output", None)
)
norm = (
    getattr(getattr(model, "model", model), "norm", None)
    or getattr(getattr(model, "transformer", None), "ln_f", None)
    or getattr(getattr(model, "gpt_neox", None), "final_layer_norm", None)
)

# 2. Extract telemetry dataframe from GlassBox flight store
telemetry_df = GlassBox.to_dataframe(last_run_id)

# 3. Build token-level data structure
device_label = device_name if "device_name" in globals() else str(DEVICE)
token_telemetry = {
    "run_id": last_run_id,
    "model_name": MODEL_NAME,
    "device": device_label,
    "prompt": TEST_PROMPT,
    "num_tokens": num_tokens,
    "tokens": [],
}

for pos in range(num_tokens):
    tok_str = tokenizer.decode([prompt_token_ids[pos]])

    # Top neurons for this token position
    pos_neurons = telemetry_df[
        (telemetry_df["token_pos"] == pos) & (telemetry_df["address"].str.startswith("N-"))
    ]
    top_neurons_list = []
    if not pos_neurons.empty:
        sorted_n = pos_neurons.sort_values(by="value", ascending=False, key=abs).head(5)
        for _, r in sorted_n.iterrows():
            addr = r["address"]
            parts = addr.split("-")
            top_neurons_list.append(
                {
                    "address": addr,
                    "layer": int(parts[1]),
                    "neuron": int(parts[2]),
                    "activation": round(float(r["value"]), 4),
                }
            )

    # Layer logit lens predictions for this token position
    layer_preds = []
    for l_idx in TRACE_LAYERS:
        if l_idx < len(hidden_states):
            h = hidden_states[l_idx][:, pos : pos + 1, :]
            h_n = norm(h) if norm is not None else h
            if lm_head is not None:
                logits = lm_head(h_n)[0, 0]
                p = torch.softmax(logits.float(), dim=-1)
                top_p, top_id = torch.topk(p, k=1)
                pred_t = tokenizer.decode([top_id.item()]).replace("\n", "\\n")
                layer_preds.append(
                    {
                        "layer": f"L-{l_idx:02d}",
                        "top_token": pred_t,
                        "prob": round(float(top_p.item()), 4),
                    }
                )

    token_telemetry["tokens"].append(
        {
            "position": pos,
            "token": tok_str,
            "token_id": prompt_token_ids[pos],
            "top_firing_neurons": top_neurons_list,
            "logit_lens_predictions": layer_preds,
        }
    )

export_dir = f"export_{last_run_id}"
os.makedirs(export_dir, exist_ok=True)

# Save token_level_telemetry.json
telemetry_file = os.path.join(export_dir, "token_level_telemetry.json")
with open(telemetry_file, "w", encoding="utf-8") as f:
    json.dump(token_telemetry, f, indent=2)

# 4. Generate general, domain-agnostic Frontier LLM Analysis Prompt
prompt_lines = [
    "# Mechanistic Interpretability Telemetry Analysis",
    "",
    f"**Model:** `{MODEL_NAME}`  ",
    f"**Hardware:** `{token_telemetry['device']}`  ",
    f"**Run ID:** `{last_run_id}`  ",
    f'**Input Prompt:** "{TEST_PROMPT}"  ',
    "**Token Sequence:** " + " ".join([f"[{tokenizer.decode([t])}]" for t in prompt_token_ids]),
    "",
    "---",
    "",
    "## Token-by-Token Internal State Progression",
    "",
]

for item in token_telemetry["tokens"]:
    pos = item["position"]
    tok = item["token"]
    tid = item["token_id"]
    prompt_lines.append(f"### Token #{pos}: {repr(tok)} (Token ID: {tid})")
    if item["top_firing_neurons"]:
        prompt_lines.append("**Top Activating Neurons:**")
        for n in item["top_firing_neurons"]:
            sign = "(+)" if n["activation"] > 0 else "(-)"
            prompt_lines.append(
                f"- Layer {n['layer']:02d}, Neuron {n['neuron']:04d} "
                f"(`{n['address']}`): `{n['activation']:+.4f}` {sign}"
            )
    else:
        prompt_lines.append("- *No traced intermediate neurons recorded at this position.*")

    if item["logit_lens_predictions"]:
        prompt_lines.append("**Logit Lens Layer Progression:**")
        preds_str = ", ".join(
            [
                f"{lp['layer']} -> {repr(lp['top_token'])} ({lp['prob'] * 100:.1f}%)"
                for lp in item["logit_lens_predictions"]
            ]
        )
        prompt_lines.append(f"- {preds_str}")
    prompt_lines.append("")

prompt_lines.extend(
    [
        "---",
        "",
        "## Analytical Tasks for Frontier LLM:",
        "You are an expert Mechanistic Interpretability researcher. "
        "Analyze the empirical token telemetry above and address:",
        "1. **Information Flow & Layer Dynamics:** At what layer depth do semantic "
        "representations stabilize or converge across the sequence?",
        "2. **Functional Neuron Attribution:** Identify which specific neurons show extreme "
        "activation magnitudes on particular tokens. What functional roles "
        "(e.g. lexical trigger, conceptual binding, syntactic delimiter) do they likely perform?",
        "3. **Circuit & Hypothesis Generation:** Based on the observed activations, hypothesize "
        "the internal circuit pathway operating here, and propose 2 counterfactual input "
        "modifications to test this circuit.",
        "",
    ]
)

llm_prompt_file = os.path.join(export_dir, "frontier_llm_prompt.md")
with open(llm_prompt_file, "w", encoding="utf-8") as f:
    f.write("\n".join(prompt_lines))

# 5. Save Benchmark Metrics
bench_file = os.path.join(export_dir, "benchmark_metrics.json")
overhead_val = round(glassbox_overhead, 2) if "glassbox_overhead" in globals() else 0.0
bench_data = {
    "model": MODEL_NAME,
    "device": token_telemetry["device"],
    "run_id": last_run_id,
    "test_prompt": TEST_PROMPT,
    "summary": results,
    "glassbox_overhead_percent": overhead_val,
}
with open(bench_file, "w", encoding="utf-8") as f:
    json.dump(bench_data, f, indent=2)

# 6. Copy Calm Viewer HTML if available
if "report_path" in globals() and os.path.exists(report_path):
    shutil.copy(report_path, os.path.join(export_dir, "calm_viewer.html"))

# 7. Package complete ZIP bundle
zip_filename = f"glassbox_artifacts_{last_run_id}.zip"
with zipfile.ZipFile(zip_filename, "w", zipfile.ZIP_DEFLATED) as zipf:
    for root, _, files in os.walk(export_dir):
        for file in files:
            full_path = os.path.join(root, file)
            arcname = os.path.relpath(full_path, export_dir)
            zipf.write(full_path, arcname)

bundle_sz = os.path.getsize(zip_filename)
print(f"Generated Frontier-LLM Telemetry Bundle: {zip_filename} ({bundle_sz} bytes)")
print(f"   Token-level telemetry: {telemetry_file}")
print(f"   Frontier LLM prompt: {llm_prompt_file}")
print(f"   Benchmark metrics: {bench_file}")

In [ ]:
# @title 💾 10. Auto-Save to Google Drive / Download ZIP Bundle
import os

from IPython.display import HTML, display

# 1. Trigger Direct Browser Download
try:
    from google.colab import files

    print("Starting browser download of artifacts bundle...")
    files.download(zip_filename)
    print("Download triggered in your browser!")
except Exception:
    print(f"Local runtime detected. Artifacts bundle is saved at: {os.path.abspath(zip_filename)}")


# 2. Function to Auto-Save to Google Drive
def save_to_google_drive(zip_path, run_id):
    """Mounts Google Drive and copies the run artifacts bundle to MyDrive/GlassBox_Runs."""
    try:
        from google.colab import drive

        drive.mount("/content/drive", force_remount=False)
        drive_dir = f"/content/drive/MyDrive/GlassBox_Runs/{run_id}"
        os.makedirs(drive_dir, exist_ok=True)
        import shutil

        shutil.copy(zip_path, os.path.join(drive_dir, os.path.basename(zip_path)))
        print(f"Successfully backed up to Google Drive: {drive_dir}/{os.path.basename(zip_path)}")
    except Exception as e:
        print(f"Google Drive sync skipped: {e}")


print("\nTo permanently save this run to your Google Drive, run this line:")
print(f"# save_to_google_drive('{zip_filename}', '{last_run_id}')")

In [ ]:
# @title 💬 11. Share Run & Benchmark to GitHub Discussions (Community Tab)
import urllib.parse

from IPython.display import HTML, display

# Prepare pre-filled GitHub Discussion
dev_str = token_telemetry["device"]
overhead_val = round(glassbox_overhead, 2) if "glassbox_overhead" in globals() else 0.0
discussion_title = f"[Community Telemetry] {dev_str} - {MODEL_NAME} ({overhead_val:+.2f}% overhead)"

first_tok = token_telemetry["tokens"][0]["token"] if token_telemetry["tokens"] else ""
last_tok = token_telemetry["tokens"][-1]["token"] if token_telemetry["tokens"] else ""

b_spd = results[0]["Throughput (tok/s)"] if len(results) > 0 else 0.0
n_spd = results[1]["Throughput (tok/s)"] if len(results) > 1 else 0.0
g_spd = results[2]["Throughput (tok/s)"] if len(results) > 2 else 0.0

slowdown_str = f"{((b_spd - n_spd) / b_spd * 100):+.1f}% slowdown" if b_spd else "N/A"

discussion_body = (
    "### GlassBox Community Benchmark & Telemetry Submission\n\n"
    f"- **Model:** `{MODEL_NAME}`\n"
    f"- **Device:** `{dev_str}`\n"
    f"- **Run ID:** `{last_run_id}`\n"
    f'- **Test Prompt:** "{TEST_PROMPT}"\n'
    f"- **Sequence Length:** {num_tokens} tokens (`{first_tok}` ... `{last_tok}`)\n\n"
    "#### Performance Benchmark\n"
    "| Condition | Throughput (tok/s) | Latency (ms/tok) | Overhead vs Baseline |\n"
    "| :--- | :--- | :--- | :--- |\n"
    f"| **Baseline PyTorch** | {b_spd:.1f} tok/s | "
    f"{1000 / b_spd if b_spd else 0:.2f} ms | Baseline |\n"
    f"| **Naive PyTorch Hook** | {n_spd:.1f} tok/s | "
    f"{1000 / n_spd if n_spd else 0:.2f} ms | {slowdown_str} |\n"
    f"| **GlassBox Async DMA** | **{g_spd:.1f} tok/s** | "
    f"**{1000 / g_spd if g_spd else 0:.2f} ms** | "
    f"**{overhead_val:+.2f}% overhead** |\n\n"
    "#### Observed Layers\n"
    f"- Traced layers: `{TRACE_LAYERS}`\n"
    f"- Provenance run seal: `{last_run_id}` (verified SHA-256 chain)\n\n"
    f"> Share your artifacts: Drag and drop your `{zip_filename}` into this post "
    "so others can explore your Calm Viewer report or analyze your token firings!\n"
)

discussion_url = (
    "https://github.com/azzouzabdelhak68-ship-it/glassbox/discussions/new?"
    f"category=general&title={urllib.parse.quote(discussion_title)}&body={urllib.parse.quote(discussion_body)}"
)

share_card_html = f'''
<div style="margin: 15px 0; padding: 18px; border: 1px solid #30363d; "
  "border-radius: 8px; background: #0d1117; color: #c9d1d9;">
  <h3 style="margin-top: 0; color: #58a6ff;">Share Your Telemetry to Community Discussions</h3>
  <p style="font-size: 14px; line-height: 1.5; color: #8b949e;">
    Click below to share your benchmark to the <b>GlassBox Discussions</b> community section.\n"
    "    No issues created, and you can drag &amp; drop your zipped bundle into your post!
  </p>
  <a href="{discussion_url}" target="_blank"
     style="display: inline-block; padding: 10px 20px; font-size: 14px; font-weight: bold; "
     "color: #fff; background-color: #238636; border-radius: 6px; text-decoration: none;">
    Open Pre-Filled GitHub Discussion Post
  </a>
</div>
'''
display(HTML(share_card_html))

## Summary & Next Steps

Congratulations! You have successfully:
- 🚀 Loaded an ultra-small open-source LLM (`Qwen/Qwen2.5-0.5B`) with zero external infrastructure.
- ⏱️ Benchmarked GlassBox's **<1% async DMA overhead** vs naive `.cpu()` hooks that cause severe GPU synchronization stalls.
- 🔭 Used the **Logit Lens** to see token predictions form across transformer layers.
- ⚡ Extracted top-activating neuron coordinates per token.
- 🎨 Explored internal activations inside the interactive **Calm Viewer**.

### Resources & Further Exploration
- 📂 **GitHub Repository**: [github.com/azzouzabdelhak68-ship-it/glassbox](https://github.com/azzouzabdelhak68-ship-it/glassbox)
- 📖 **Documentation**: Read the [Architecture Guide](file:///docs/ARCHITECTURE.md) and [Adapter Guide](file:///docs/ADAPTER_GUIDE.md).
- 💻 **CLI Usage**: Try `glassbox find` and `glassbox compare` on sealed flight runs.